# Shooting Method for the Blasius Boundary-Layer Equation
**AE801 Scientific Computing Lab-1 — Numerical Solution of ODEs (BVP)**

## 1. Problem statement

Steady, incompressible flow of a uniform stream $U_\infty$ over a flat plate reduces, under the similarity variable
$\eta = y\sqrt{U_\infty/(\nu x)}$ with $u/U_\infty = f'(\eta)$, to the **Blasius equation**

$$
f''' + \tfrac{1}{2} f f'' = 0, \qquad 0 < \eta < \infty,
$$

with boundary conditions

$$
f(0) = 0,\qquad f'(0) = 0 \quad(\text{no slip, no suction}),\qquad f'(\eta)\to 1 \text{ as } \eta\to\infty \quad(\text{free stream}).
$$

This is a nonlinear two-point boundary-value problem (BVP) on a semi-infinite domain. We solve it with the
**shooting method**: convert it to an initial-value problem (IVP) with one unknown initial condition, integrate with
explicit Runge–Kutta schemes (RK3 and RK4), and adjust the unknown until the far-field condition is satisfied.

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

np.set_printoptions(precision=10)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

## 2. Step (a) — Reduction to a first-order system

Define the state vector $Y = (y_1, y_2, y_3)^T = (f,\ f',\ f'')^T$. Then

$$
Y' = G(Y) = \begin{pmatrix} y_2 \\ y_3 \\ -\tfrac12\, y_1 y_3 \end{pmatrix}.
$$

The system is **autonomous** (no explicit $\eta$ dependence), so $G$ only needs $Y$.

In [13]:
def G(Y):
    f, fp, fpp = Y
    return np.array([fp, fpp, -0.5 * f * fpp])

## 3. Step (a) — Fixed-step Runge–Kutta integrators

**RK3 (Kutta's classical third-order scheme)**, step $h=\Delta\eta$:

$$
\begin{aligned}
k_1 &= G(Y_n), \\
k_2 &= G\!\left(Y_n + \tfrac{h}{2}k_1\right), \\
k_3 &= G\!\left(Y_n - h k_1 + 2h k_2\right), \\
Y_{n+1} &= Y_n + \tfrac{h}{6}\left(k_1 + 4k_2 + k_3\right).
\end{aligned}
$$

Local truncation error $\mathcal O(h^4)$, global error $\mathcal O(h^3)$.

**RK4 (classical fourth-order scheme)**:

$$
\begin{aligned}
k_1 &= G(Y_n), & k_2 &= G\!\left(Y_n + \tfrac{h}{2}k_1\right), \\
k_3 &= G\!\left(Y_n + \tfrac{h}{2}k_2\right), & k_4 &= G\!\left(Y_n + h k_3\right), \\
Y_{n+1} &= Y_n + \tfrac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right).
\end{aligned}
$$

Global error $\mathcal O(h^4)$.

The integrator below takes $N = \eta_\infty/\Delta\eta$ steps. $\eta_\infty$ is chosen as an integer multiple of every
$\Delta\eta$ used, so the last grid point lands exactly on $\eta_\infty$ (no partial final step that would spoil the

In [14]:
def rk3_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y - h * k1 + 2.0 * h * k2)
    return Y + h / 6.0 * (k1 + 4.0 * k2 + k3)

def rk4_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y + 0.5 * h * k2)
    k4 = G(Y + h * k3)
    return Y + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

STEPPERS = {"RK3": rk3_step, "RK4": rk4_step}

def integrate(s, h, eta_inf, method="RK4", store=False):
    # Integrate the IVP Y(0) = (0, 0, s) from eta = 0 to eta_inf with fixed step h.
    N = int(round(eta_inf / h))
    assert abs(N * h - eta_inf) < 1e-12, "eta_inf must be an integer multiple of h"
    step = STEPPERS[method]
    Y = np.array([0.0, 0.0, s])
    if store:
        sol = np.empty((N + 1, 3)); sol[0] = Y
    for n in range(N):
        Y = step(Y, h)
        if store:
            sol[n + 1] = Y
    if store:
        return np.linspace(0.0, eta_inf, N + 1), sol
    return Y

In [15]:
def rk3_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y - h * k1 + 2.0 * h * k2)
    return Y + h / 6.0 * (k1 + 4.0 * k2 + k3)

def rk4_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y + 0.5 * h * k2)
    k4 = G(Y + h * k3)
    return Y + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

STEPPERS = {"RK3": rk3_step, "RK4": rk4_step}

def integrate(s, h, eta_inf, method="RK4", store=False):
    # Integrate the IVP Y(0) = (0, 0, s) from eta = 0 to eta_inf with fixed step h.
    N = int(round(eta_inf / h))
    assert abs(N * h - eta_inf) < 1e-12, "eta_inf must be an integer multiple of h"
    step = STEPPERS[method]
    Y = np.array([0.0, 0.0, s])
    if store:
        sol = np.empty((N + 1, 3)); sol[0] = Y
    for n in range(N):
        Y = step(Y, h)
        if store:
            sol[n + 1] = Y
    if store:
        return np.linspace(0.0, eta_inf, N + 1), sol
    return Y

In [16]:
def rk3_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y - h * k1 + 2.0 * h * k2)
    return Y + h / 6.0 * (k1 + 4.0 * k2 + k3)

def rk4_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y + 0.5 * h * k2)
    k4 = G(Y + h * k3)
    return Y + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

STEPPERS = {"RK3": rk3_step, "RK4": rk4_step}

def integrate(s, h, eta_inf, method="RK4", store=False):
    # Integrate the IVP Y(0) = (0, 0, s) from eta = 0 to eta_inf with fixed step h.
    N = int(round(eta_inf / h))
    assert abs(N * h - eta_inf) < 1e-12, "eta_inf must be an integer multiple of h"
    step = STEPPERS[method]
    Y = np.array([0.0, 0.0, s])
    if store:
        sol = np.empty((N + 1, 3)); sol[0] = Y
    for n in range(N):
        Y = step(Y, h)
        if store:
            sol[n + 1] = Y
    if store:
        return np.linspace(0.0, eta_inf, N + 1), sol
    return Y

In [17]:
def rk3_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y - h * k1 + 2.0 * h * k2)
    return Y + h / 6.0 * (k1 + 4.0 * k2 + k3)

def rk4_step(Y, h):
    k1 = G(Y)
    k2 = G(Y + 0.5 * h * k1)
    k3 = G(Y + 0.5 * h * k2)
    k4 = G(Y + h * k3)
    return Y + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

STEPPERS = {"RK3": rk3_step, "RK4": rk4_step}

def integrate(s, h, eta_inf, method="RK4", store=False):
    # Integrate the IVP Y(0) = (0, 0, s) from eta = 0 to eta_inf with fixed step h.
    N = int(round(eta_inf / h))
    assert abs(N * h - eta_inf) < 1e-12, "eta_inf must be an integer multiple of h"
    step = STEPPERS[method]
    Y = np.array([0.0, 0.0, s])
    if store:
        sol = np.empty((N + 1, 3)); sol[0] = Y
    for n in range(N):
        Y = step(Y, h)
        if store:
            sol[n + 1] = Y
    if store:
        return np.linspace(0.0, eta_inf, N + 1), sol
    return Y

In [18]:
def check_order(stepper, hs, T=1.0):
    errs = []
    for h in hs:
        Y = np.array([1.0, 1.0, 1.0])
        # temporarily integrate y' = A y with A = companion matrix of y''' = y
        for _ in range(int(round(T / h))):
            Gl = lambda Z: np.array([Z[1], Z[2], Z[0]])
            if stepper == "RK3":
                k1 = Gl(Y); k2 = Gl(Y + 0.5*h*k1); k3 = Gl(Y - h*k1 + 2*h*k2)
                Y = Y + h/6*(k1 + 4*k2 + k3)
            else:
                k1 = Gl(Y); k2 = Gl(Y + 0.5*h*k1); k3 = Gl(Y + 0.5*h*k2); k4 = Gl(Y + h*k3)
                Y = Y + h/6*(k1 + 2*k2 + 2*k3 + k4)
        errs.append(abs(Y[0] - np.e))
    return np.array(errs)

hs = np.array([0.1, 0.05, 0.025, 0.0125])
for m in ("RK3", "RK4"):
    e = check_order(m, hs)
    print(f"{m}: error ratios = {np.round(e[:-1]/e[1:], 2)}  -> observed order ≈ {np.round(np.log2(e[:-1]/e[1:]), 3)}")

RK3: error ratios = [7.69 7.84 7.92]  -> observed order ≈ [2.942 2.971 2.986]
RK4: error ratios = [15.35 15.67 15.83]  -> observed order ≈ [3.94  3.97  3.985]
